In [1]:
from pathlib import Path
import pandas as pd

# Resolve dataset path reliably whether notebook runs from /notebooks or project root
path = Path("../dataset/ml-1m") if Path("../dataset/ml-1m").exists() else Path("dataset/ml-1m")

ratings = pd.read_csv(
    path / "ratings.dat",
    sep="::",
    engine="python",
    names=["userId", "movieId", "rating", "timestamp"],
    encoding="latin1"
)

movies = pd.read_csv(
    path / "movies.dat",
    sep="::",
    engine="python",
    names=["movieId", "title", "genres"],
    encoding="latin1"
)

users = pd.read_csv(
    path / "users.dat",
    sep="::",
    engine="python",
    names=["userId", "gender", "age", "occupation", "zip"],
    encoding="latin1"
)

print("Loaded successfully!")
print(ratings.head())
print(movies.head())
print(users.head())

Loaded successfully!
   userId  movieId  rating  timestamp
0       1     1193       5  978300760
1       1      661       3  978302109
2       1      914       3  978301968
3       1     3408       4  978300275
4       1     2355       5  978824291
   movieId                               title                        genres
0        1                    Toy Story (1995)   Animation|Children's|Comedy
1        2                      Jumanji (1995)  Adventure|Children's|Fantasy
2        3             Grumpier Old Men (1995)                Comedy|Romance
3        4            Waiting to Exhale (1995)                  Comedy|Drama
4        5  Father of the Bride Part II (1995)                        Comedy
   userId gender  age  occupation    zip
0       1      F    1          10  48067
1       2      M   56          16  70072
2       3      M   25          15  55117
3       4      M   45           7  02460
4       5      M   25          20  55455


In [2]:
import pandas as pd

# Create user-movie matrix
user_movie_matrix = ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

print(user_movie_matrix.shape)

(6040, 3706)


In [3]:
import pandas as pd

# Create user-movie matrix
user_movie_matrix = ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

user_movie_matrix.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
userId,,,,,,,,,,,,,,,,,,,,,
1,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
from sklearn.metrics.pairwise import cosine_similarity

# Fill NaN with 0
user_movie_filled = user_movie_matrix.fillna(0)

# Compute similarity between users
user_similarity = cosine_similarity(user_movie_filled)

# Convert to DataFrame
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index
)

user_similarity_df.head()

userId,1,2,3,4,5,6,7,8,9,10,...,6031,6032,6033,6034,6035,6036,6037,6038,6039,6040
userId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.096382,0.120610,0.132455,0.090158,0.179222,0.059678,0.138241,0.226148,0.255288,...,0.170588,0.082006,0.069807,0.033663,0.114877,0.186329,0.135979,0.000000,0.174604,0.133590
2,0.096382,1.000000,0.151479,0.171176,0.114394,0.100865,0.305787,0.203337,0.190198,0.226861,...,0.112503,0.091222,0.268565,0.014286,0.183384,0.228241,0.206274,0.066118,0.066457,0.218276
3,0.120610,0.151479,1.000000,0.151227,0.062907,0.074603,0.138332,0.077656,0.126457,0.213655,...,0.092960,0.125864,0.161507,0.000000,0.097308,0.143264,0.107744,0.120234,0.094675,0.133144
4,0.132455,0.171176,0.151227,1.000000,0.045094,0.013529,0.130339,0.100856,0.093651,0.120738,...,0.163629,0.093041,0.382803,0.000000,0.082097,0.170583,0.127464,0.062907,0.064634,0.137968
5,0.090158,0.114394,0.062907,0.045094,1.000000,0.047449,0.126257,0.220817,0.261330,0.117052,...,0.100652,0.035732,0.061806,0.054151,0.179083,0.293365,0.172686,0.020459,0.027689,0.241437


In [5]:
def recommend_user_based(user_id, top_n=5):
    
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:6]
    
    similar_user_ids = similar_users.index
    
    movies_rated_by_similar = ratings[
        ratings["userId"].isin(similar_user_ids)
    ]
    
    recommended_movies = movies_rated_by_similar.groupby("movieId")["rating"].mean()
    
    recommended_movies = recommended_movies.sort_values(ascending=False)
    
    return movies[movies["movieId"].isin(recommended_movies.head(top_n).index)][["title", "genres"]]
# Example usage
print(recommend_user_based(1, top_n=5))

                     title                        genres
38         Clueless (1995)                Comedy|Romance
2890     Fight Club (1999)                         Drama
3045    Toy Story 2 (1999)   Animation|Children's|Comedy
3090  Fantasia 2000 (1999)  Animation|Children's|Musical
3184  Wayne's World (1992)                        Comedy


In [6]:
# Split genres
movies["genres"] = movies["genres"].str.split("|")

movies.head()
def recommend_content_based(user_id, top_n=5):
    
    user_movies = ratings[ratings["userId"] == user_id]
    liked_movies = user_movies[user_movies["rating"] >= 4]
    
    liked_movie_ids = liked_movies["movieId"]
    
    liked_genres = movies[movies["movieId"].isin(liked_movie_ids)]["genres"]
    
    all_genres = sum(liked_genres, [])
    
    genre_counts = pd.Series(all_genres).value_counts()
    
    top_genres = genre_counts.index[:3]
    
    recommendations = movies[
        movies["genres"].apply(lambda x: any(g in x for g in top_genres))
    ]
    
    return recommendations.head(top_n)[["title", "genres"]]
# Example usage
print(recommend_content_based(1))

                             title                            genres
0                 Toy Story (1995)   [Animation, Children's, Comedy]
1                   Jumanji (1995)  [Adventure, Children's, Fantasy]
3         Waiting to Exhale (1995)                   [Comedy, Drama]
7              Tom and Huck (1995)           [Adventure, Children's]
10  American President, The (1995)          [Comedy, Drama, Romance]


In [7]:
def collaborative_scores(user_id):
    
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:11]
    similar_user_ids = similar_users.index
    
    sim_scores = similar_users.values
    
    movies_rated = ratings[ratings["userId"].isin(similar_user_ids)]
    
    movie_scores = {}
    
    for idx, row in movies_rated.iterrows():
        movie_id = row["movieId"]
        rating = row["rating"]
        sim_user = row["userId"]
        
        similarity = user_similarity_df.loc[user_id, sim_user]
        
        if movie_id not in movie_scores:
            movie_scores[movie_id] = 0
        
        movie_scores[movie_id] += similarity * rating
    
    return pd.DataFrame(movie_scores.items(), columns=["movieId", "cf_score"])


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

movies["genres_str"] = movies["genres"].apply(lambda x: " ".join(x))

tfidf = TfidfVectorizer()
genre_matrix = tfidf.fit_transform(movies["genres_str"])

genre_similarity = cosine_similarity(genre_matrix)

In [9]:
def content_scores(user_id):
    
    user_movies = ratings[ratings["userId"] == user_id]
    liked_movies = user_movies[user_movies["rating"] >= 4]
    
    liked_ids = liked_movies["movieId"].values
    
    scores = {}
    
    for movie_id in liked_ids:
        idx = movies[movies["movieId"] == movie_id].index[0]
        
        sim_scores = list(enumerate(genre_similarity[idx]))
        
        for i, score in sim_scores:
            target_movie_id = movies.iloc[i]["movieId"]
            
            if target_movie_id not in scores:
                scores[target_movie_id] = 0
                
            scores[target_movie_id] += score
    
    return pd.DataFrame(scores.items(), columns=["movieId", "content_score"])

In [10]:
def hybrid_recommend(user_id, top_n=10):
    
    cf = collaborative_scores(user_id)
    content = content_scores(user_id)
    
    # Merge both scores
    hybrid = pd.merge(cf, content, on="movieId", how="outer").fillna(0)
    
    # Normalize
    hybrid["cf_score"] = hybrid["cf_score"] / hybrid["cf_score"].max()
    hybrid["content_score"] = hybrid["content_score"] / hybrid["content_score"].max()
    
    # Final weighted score
    hybrid["final_score"] = 0.7 * hybrid["cf_score"] + 0.3 * hybrid["content_score"]
    
    # Remove already rated movies
    rated = ratings[ratings["userId"] == user_id]["movieId"]
    hybrid = hybrid[~hybrid["movieId"].isin(rated)]
    
    hybrid = hybrid.sort_values("final_score", ascending=False)
    
    return pd.merge(
        hybrid.head(top_n),
        movies[["movieId", "title", "genres"]],
        on="movieId"
    )
# Example usage
print(hybrid_recommend(1, top_n=5))

   movieId  cf_score  content_score  final_score                       title  \
0     2081  0.878376       0.960833     0.903114  Little Mermaid, The (1989)   
1     2078  0.739194       0.992425     0.815163     Jungle Book, The (1967)   
2     2096  0.641779       0.938350     0.730750      Sleeping Beauty (1959)   
3      364  0.641389       0.938350     0.730477       Lion King, The (1994)   
4     1282  0.598068       0.938350     0.700152             Fantasia (1940)   

                                              genres  
0  [Animation, Children's, Comedy, Musical, Romance]  
1           [Animation, Children's, Comedy, Musical]  
2                   [Animation, Children's, Musical]  
3                   [Animation, Children's, Musical]  
4                   [Animation, Children's, Musical]  
